## 1. Import Dependencies

In [1]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

## 2. Hyperparameters

In [2]:
DATA_DIRECT = "./dataset3" 
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 30
LEARNING_RATE = 3e-4
MODEL_PATH = "model/resnet18_emotion.pth"
WEIGHT_DECAY = 1e-4

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## 3. Model Definition with ResNet18

In [4]:
class EmotionResNet(nn.Module):
    def __init__(self, num_classes=7, freeze_backbone=True):
        super().__init__()
        self.backbone = models.resnet18(pretrained=True)
        
        if freeze_backbone:
            for param in list(self.backbone.parameters())[:-30]:
                param.requires_grad = False
        
        # Replace final fully connected layer
        num_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(num_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)

## 4. Data Transforms with Augmentation

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) 
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

## 5. Load and Prepare Dataset

In [6]:
train_dataset = datasets.ImageFolder('./dataset3/train', transform=train_transform)
test_dataset = datasets.ImageFolder('./dataset3/test', transform=test_transform)

class_names = train_dataset.classes
num_classes = len(class_names)

print(f"Classes: {class_names}")

print(f"Number of classes: {num_classes}")
print(f"Test images: {len(test_dataset)}")
print(f"Training images: {len(train_dataset)}")

Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Number of classes: 7
Test images: 7178
Training images: 28709


In [7]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                        num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, 
                        num_workers=4, pin_memory=True)

print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

Training batches: 898
Test batches: 225


## 6. Initialize Model and Training Components

In [8]:
model = EmotionResNet(num_classes=num_classes, freeze_backbone=True).to(device)

# Count trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")

d:\HERDIN\CODE\Python\Emotion Recognition\venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\HERDIN\CODE\Python\Emotion Recognition\venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Trainable parameters: 10,331,911 / 11,310,151


In [9]:
# Compute class weights for imbalanced data
train_labels = np.array([label for _, label in train_dataset])
class_counts = np.bincount(train_labels)
class_weights = 1.0 / (class_counts + 1e-6)
class_weights = class_weights / class_weights.sum() * len(class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)

print("Class distribution (training set):")
for i, (class_name, count) in enumerate(zip(class_names, class_counts)):
    print(f"  {class_name}: {count} images (weight: {class_weights[i]:.3f})")

Class distribution (training set):
  angry: 3995 images (weight: 0.480)
  disgust: 436 images (weight: 4.398)
  fear: 4097 images (weight: 0.468)
  happy: 7215 images (weight: 0.266)
  neutral: 4965 images (weight: 0.386)
  sad: 4830 images (weight: 0.397)
  surprise: 3171 images (weight: 0.605)


In [9]:
# Loss function, optimizer, and scheduler
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Scheduler for better convergence
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, 
    max_lr=LEARNING_RATE, 
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS
)

# Mixed precision training
scaler = torch.amp.GradScaler('cuda' if torch.cuda.is_available() else 'cpu')

NameError: name 'class_weights' is not defined

## 7. Training Function

In [11]:
def train():
    best_test_acc = 0.0
    patience = 7
    no_improve_epochs = 0
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
    
    print("\n" + "="*70)
    print("STARTING TRAINING WITH RESNET18")
    print("="*70 + "\n")
    
    for epoch in range(1, EPOCHS + 1):
        # Training phase
        model.train()
        running_loss = 0.0
        running_correct = 0
        total = 0
        t0 = time.time()
        
        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            optimizer.zero_grad()
            
            with torch.amp.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            # Backward
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            
            # Track metrics
            running_loss += loss.item() * images.size(0)
            predict = outputs.argmax(dim=1)
            running_correct += (predict == labels).sum().item()
            total += images.size(0)
        
        train_loss = running_loss / total
        train_acc = running_correct / total
        
        # Validation
        model.eval()
        test_loss = 0.0
        test_correct = 0
        test_total = 0
        
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                
                with torch.amp.autocast(device_type='cuda' if torch.cuda.is_available() else 'cpu'):
                    outputs = model(images)
                    loss = criterion(outputs, labels)
                
                test_loss += loss.item() * images.size(0)
                predict = outputs.argmax(dim=1)
                test_correct += (predict == labels).sum().item()
                test_total += images.size(0)
        
        test_loss = test_loss / test_total
        test_acc = test_correct / test_total
        
        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        
        elapsed = time.time() - t0
        current_lr = optimizer.param_groups[0]['lr']
        
        print(f"Epoch {epoch:2d}/{EPOCHS} | "
            f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
            f"Test Loss: {test_loss:.4f}, Acc: {test_acc:.4f} | "
            f"LR: {current_lr:.6f} | Time: {elapsed:.1f}s")
        
        # Save best model
        if test_acc > best_test_acc:
            best_test_acc = test_acc
            no_improve_epochs = 0
            torch.save({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'class_names': class_names,
                'img_size': IMG_SIZE,
                'test_acc': best_test_acc,
                'test_loss': test_loss
            }, MODEL_PATH)
            print(f"  ✓ SAVED BEST MODEL (test_acc: {best_test_acc:.4f})")
        else:
            no_improve_epochs += 1
            if no_improve_epochs >= patience:
                print(f"\n⚠ Early stopping at epoch {epoch} - no improvement for {patience} epochs")
                break
    
    print("\n" + "="*70)
    print("TRAINING FINISHED!")
    print(f"   Best test accuracy: {best_test_acc:.4f} ({best_test_acc*100:.2f}%)")
    print("="*70 + "\n")
    
    return history, best_test_acc

## 8. Evaluation and Analysis

In [ ]:
def evaluate_model(model_path=MODEL_PATH):
    # Load best model
    checkpoint = torch.load(model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state'])
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    # Confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.title('Confusion Matrix - ResNet18 Emotion Detection', fontsize=14)
    plt.tight_layout()
    plt.savefig('confusion_matrix_resnet18.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Classification report
    print("\nClassification Report:")
    print("="*70)
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))
    
    return cm

In [13]:
def plot_training_history(history):
    """Plot training and validation metrics"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss plot
    ax1.plot(history['train_loss'], label='Train Loss', marker='o')
    ax1.plot(history['test_loss'], label='Test Loss', marker='s')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Test Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Accuracy plot
    ax2.plot(history['train_acc'], label='Train Accuracy', marker='o')
    ax2.plot(history['test_acc'], label='Test Accuracy', marker='s')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Training and Test Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('training_history_resnet18.png', dpi=300, bbox_inches='tight')
    plt.show()

## 9. Webcam Inference

In [6]:
# Pre-trained gender and age model
GENDER_MODEL = './model/gender_net.caffemodel'
GENDER_PROTO = './model/gender_deploy.prototxt'
AGE_MODEL = './model/age_net.caffemodel'
AGE_PROTO = './model/age_deploy.prototxt'

In [10]:
def infer_from_webcam(checkpoint_path=MODEL_PATH, use_gender=True, use_age=True, 
                    use_smoothing=True, history_size=7, min_confidence=0.50):
    """
    Controls:
        'q' - Quit
        'f' - Toggle fullscreen
        'g' - Toggle gender detection on/off
        'a' - Toggle age detection on/off
        'ESC' - Exit fullscreen
    """
    if not os.path.exists(checkpoint_path):
        print(f"Model not found at {checkpoint_path}")
        print("Please train the model first by running the training cell!")
        return
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    class_names_local = checkpoint['class_names']
    num_classes = len(class_names_local)
    
    model_local = EmotionResNet(num_classes=num_classes, freeze_backbone=True).to(device)
    model_local.load_state_dict(checkpoint['model_state'])
    model_local.eval()
    
    print(f"Loaded emotion model with test accuracy: {checkpoint['test_acc']:.4f}")
    
    gender_net = None
    gender_available = os.path.exists(GENDER_PROTO) and os.path.exists(GENDER_MODEL)
    
    if gender_available:
        try:
            gender_net = cv2.dnn.readNetFromCaffe(GENDER_PROTO, GENDER_MODEL)
            print("Gender detection model loaded")
        except Exception as e:
            print(f"Could not load gender model: {e}")
            gender_available = False
    else:
        print("Gender model files not found")
        
    age_net = None
    age_available = os.path.exists(AGE_PROTO) and os.path.exists(AGE_MODEL)
    
    if age_available:
        try:
            age_net = cv2.dnn.readNetFromCaffe(AGE_PROTO, AGE_MODEL)
            print("Age detection model loaded")
        except Exception as e:
            print(f"Could not load age model: {e}")
            age_available = False
    else:
        print("Age model files not found")
    
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    
    print("\n" + "="*70)
    print("STARTING WEBCAM - ResNet18 Emotion Detection")
    if gender_available:
        print("Gender detection: Press 'g' to toggle")
    if age_available:
        print("Age detection: Press 'a' to toggle")
    print("="*70)
    print("Controls:")
    print("  - Press 'q' to quit")
    print("  - Press 'f' for fullscreen")
    if gender_available:
        print("  - Press 'g' to toggle gender detection")
    if age_available:
        print("  - Press 'a' to toggle age detection")
    print("  - Press 'ESC' to exit fullscreen")
    print("="*70 + "\n")
    
    cam = cv2.VideoCapture(1)
    if not cam.isOpened():
        print("Cannot open webcam")
        return

    cam.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cam.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    cam.set(cv2.CAP_PROP_FPS, 30)
    
    window_name = "ResNet18 Emotion + Gender + Age Detection"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, 1280, 720)
    
    frame_count = 0
    last_predictions = {}
    fullscreen = False
    fps_history = []
    gender_enabled = use_gender and gender_available
    age_enabled = use_age and age_available
    gender_labels = ['Male', 'Female']
    age_labels = ['(0-2)', '(4-6)', '(8-12)', '(15-20)', '(25-32)', '(38-43)', '(48-53)', '(60-100)']
    
    MODEL_MEAN_VALUES = (78.4263377603, 87.7689143744, 114.895847746)

    from collections import deque
    prediction_history = {}  
    stable_predictions = {}  
    
    while True:
        ret, frame = cam.read()
        if not ret:
            break
        
        start_time = time.time()

        if frame_count % 2 == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, 1.3, 5, minSize=(50, 50))
            
            current_predictions = {}
            for i, (x, y, w, h) in enumerate(faces):
                face_roi = frame[y:y+h, x:x+w]

                face_pil = Image.fromarray(cv2.cvtColor(face_roi, cv2.COLOR_BGR2RGB))
                tensor = test_transform(face_pil).unsqueeze(0).to(device)
                
                with torch.no_grad():
                    outputs = model_local(tensor)
                    probs = torch.nn.functional.softmax(outputs, dim=1)
                    confidence, predicted = torch.max(probs, 1)
                    
                    emotion = class_names_local[predicted.item()]
                    conf_score = confidence.item()
                
                gender = None
                gender_conf = 0.0
                if gender_enabled and gender_net is not None:
                    try:
                        face_blob = cv2.dnn.blobFromImage(
                            face_roi, 1.0, (227, 227), 
                            MODEL_MEAN_VALUES, swapRB=False
                        )
                        gender_net.setInput(face_blob)
                        gender_preds = gender_net.forward()
                        gender_idx = gender_preds[0].argmax()
                        gender = gender_labels[gender_idx]
                        gender_conf = gender_preds[0][gender_idx]
                    except:
                        gender = None
                
                age = None
                age_conf = 0.0
                if age_enabled and age_net is not None:
                    try:
                        face_blob = cv2.dnn.blobFromImage(
                            face_roi, 1.0, (227, 227), 
                            MODEL_MEAN_VALUES, swapRB=False
                        )
                        age_net.setInput(face_blob)
                        age_preds = age_net.forward()
                        age_idx = age_preds[0].argmax()
                        age = age_labels[age_idx]
                        age_conf = age_preds[0][age_idx]
                    except:
                        age = None
                
                # Apply temporal smoothing
                if use_smoothing:
                    if i not in prediction_history:
                        prediction_history[i] = deque(maxlen=history_size)
                        stable_predictions[i] = emotion
                    
                    prediction_history[i].append((emotion, conf_score))
                    
                    if conf_score >= min_confidence:
                        if len(prediction_history[i]) >= 3:
                            emotions_in_history = [e for e, c in prediction_history[i]]
                            stable_emotion = max(set(emotions_in_history), 
                                                key=emotions_in_history.count)
                            stable_predictions[i] = stable_emotion
                        else:
                            stable_predictions[i] = emotion
                    
                    display_emotion = stable_predictions[i]
                else:
                    display_emotion = emotion
                
                current_predictions[i] = {
                    'bbox': (x, y, w, h),
                    'emotion': display_emotion,
                    'confidence': conf_score,
                    'gender': gender,
                    'gender_conf': gender_conf,
                    'age': age,
                    'age_conf': age_conf
                }
            
            if use_smoothing:
                active_face_ids = set(current_predictions.keys())
                for face_id in list(prediction_history.keys()):
                    if face_id not in active_face_ids:
                        del prediction_history[face_id]
                        if face_id in stable_predictions:
                            del stable_predictions[face_id]
            
            last_predictions = current_predictions
        
        # Draw results
        height, width = frame.shape[:2]
        font_scale = max(0.6, min(width/1200, height/700))
        thickness = max(2, int(font_scale * 2))
        
        for i, pred in last_predictions.items():
            x, y, w, h = pred['bbox']
            
            if pred['confidence'] > 0.7:
                color = (0, 255, 0)  # High confidence
            elif pred['confidence'] > 0.5:
                color = (0, 255, 255)  # Medium confidence
            else:
                color = (0, 165, 255)  # Low confidence
            
            # Bounding box
            cv2.rectangle(frame, (x, y), (x+w, y+h), color, 3)
            
            label_parts = []
            if gender_enabled and pred['gender']:
                label_parts.append(pred['gender'])
            if age_enabled and pred['age']:
                label_parts.append(pred['age'])
            label_parts.append(f"{pred['emotion']}: (confidence){pred['confidence']*100:.1f}%")
            
            label = " | ".join(label_parts)
            text_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)[0]
            
            cv2.rectangle(frame, (x, y-text_size[1]-15), 
                        (x + text_size[0] + 10, y), color, -1)
            cv2.putText(frame, label, (x + 5, y-8), 
                        cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), thickness)
            
            bar_width = w
            bar_height = 8
            confidence_width = int(bar_width * pred['confidence'])
            cv2.rectangle(frame, (x, y+h+5), (x+bar_width, y+h+5+bar_height), (50, 50, 50), -1)
            cv2.rectangle(frame, (x, y+h+5), (x+confidence_width, y+h+5+bar_height), color, -1)

        elapsed = time.time() - start_time
        if elapsed > 0:
            fps = 1.0 / elapsed
            fps_history.append(fps)
            if len(fps_history) > 30:
                fps_history.pop(0)
        
        avg_fps = sum(fps_history) / len(fps_history) if fps_history else 0.0
        
        status_parts = [f"Faces: {len(last_predictions)}"]
        if gender_available:
            status_parts.append(f"Gender: {'ON' if gender_enabled else 'OFF'}")
        if age_available:
            status_parts.append(f"Age: {'ON' if age_enabled else 'OFF'}")
        
        status_text = " | ".join(status_parts)
        cv2.putText(frame, status_text, (15, 35), 
                cv2.FONT_HERSHEY_SIMPLEX, font_scale*0.8, (255, 255, 255), thickness)

        controls_parts = ["q=quit", "f=fullscreen"]
        if gender_available:
            controls_parts.append("g=toggle gender")
        if age_available:
            controls_parts.append("a=toggle age")
        controls_parts.append("ESC=exit fullscreen")
        controls = " | ".join(controls_parts)
        
        cv2.putText(frame, controls, (15, height-20), 
                cv2.FONT_HERSHEY_SIMPLEX, font_scale*0.6, (200, 200, 200), max(1, thickness-1))
        
        cv2.imshow(window_name, frame)
        
        key = cv2.waitKey(1) & 0xFF
        
        if key == ord('q'):
            break
        elif key == ord('f'):
            fullscreen = not fullscreen
            if fullscreen:
                cv2.setWindowProperty(window_name, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)
                print("Fullscreen mode ON")
            else:
                cv2.setWindowProperty(window_name, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_NORMAL)
                print("Fullscreen mode OFF")
        elif key == ord('g'):
            if gender_available:
                gender_enabled = not gender_enabled
                print(f"Gender detection: {'ON' if gender_enabled else 'OFF'}")
            else:
                print("Gender model not available")
        elif key == ord('a'):
            if age_available:
                age_enabled = not age_enabled
                print(f"Age detection: {'ON' if age_enabled else 'OFF'}")
            else:
                print("Age model not available")
        elif key == 27:
            if fullscreen:
                cv2.setWindowProperty(window_name, cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_NORMAL)
                fullscreen = False
                print("Exited fullscreen mode")
        
        frame_count += 1
    
    cam.release()
    cv2.destroyAllWindows()

## 10. Run Training

In [11]:
# Train the model
# history, best_acc = train()

## 11. Visualize Results

In [12]:
# Plot train history
# plot_training_history(history)

In [13]:
# Evaluate and show confusion matrix
# cm = evaluate_model()

## 12. Run Webcam Detection

In [15]:
# Run
infer_from_webcam()

C:\Users\acer\AppData\Local\Temp\ipykernel_12952\1111763541.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=device

Loaded emotion model with test accuracy: 0.7236
Gender detection model loaded
Age detection model loaded

STARTING WEBCAM - ResNet18 Emotion Detection
Gender detection: Press 'g' to toggle
Age detection: Press 'a' to toggle
Controls:
  - Press 'q' to quit
  - Press 'f' for fullscreen
  - Press 'g' to toggle gender detection
  - Press 'a' to toggle age detection
  - Press 'ESC' to exit fullscreen

